# Visualising AGN feedback (jet-mode outflows) in SIMBA

**What this notebook does, and the physics behind it.**

SIMBA does *not* hydrodynamically resolve relativistic radio jets. AGN feedback is
**sub-grid kinetic**: when a black hole is in *jet mode* (low Eddington ratio,
`f_Edd ~< 0.2`, and `M_BH ~> 10^7.5 Msun`) gas particles near the BH are kicked
**bipolarly along the BH angular-momentum axis** at up to ~8000 km/s and heated
toward the halo virial temperature. The particles are temporarily hydro-decoupled.

So what physically exists in the snapshot is **fast, hot, bipolar outflowing gas**
propagating into the CGM. That is the signature we image here:

| Diagnostic | What the feedback looks like |
|---|---|
| **Signed radial velocity** `v.r_hat` (edge-on) | Bipolar **red** (outflow) lobes along the minor axis |
| **Temperature** (edge-on) | Hot plumes / shells perpendicular to the disk |
| **Gas surface density** | Evacuated bipolar **cavities** |
| **Velocity streamlines** | Streamlines diverging out of the disk plane |

**Honest limitations**
- You will see bipolar outflows / hot lobes / cavities, *not* a collimated jet
  morphology with internal shocks. That structure is not in the simulation.
- Jet-mode episodes are intermittent. We rank galaxies by BH mass / Eddington
  ratio to find one *currently* launching, but you may need to try a few snapshots.
- We follow the codebase convention for the velocity scale-factor handling
  (`* scale_factor`, as in `SingleRender.stream_plot`). Sign and direction —
  what matters for *seeing* the jet — are unaffected.

> "Real time": the last section caches the particle arrays once and gives you an
> **interactive rotation slider** (re-projection only, no reload) so you can spin
> the galaxy and watch the bipolar lobes. There is also an optional rotation-GIF
> cell and a multi-snapshot evolution sketch.

## 1. Imports & data load

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm

import yt
import caesar
import sphviewer as sph
from sphviewer.tools import QuickView

from simbanator.io.simba import Simulation
# Reuse the codebase's viewing-geometry helpers
from simbanator.visualization.rendering import (
    find_rot_ax,
    rotation_matrices_from_angles,
    SingleRender,
)

plt.rcParams.update({
    "savefig.facecolor": "w", "figure.facecolor": "w",
    "font.size": 16, "axes.labelsize": 16,
    "xtick.labelsize": 12, "ytick.labelsize": 12,
})

In [ ]:
# --- Choose snapshot ----------------------------------------------------
SNAP = 149                 # change me
sb       = Simulation('cis25')
snapfile = sb.get_sim_file(SNAP)
catfile  = sb.get_caesar_file(SNAP)
try:
    z_snap = sb.get_z_from_snap(SNAP)
except Exception:
    z_snap = float('nan')

print(f"Snapshot : {snapfile}")
print(f"Catalog  : {catfile}")
print(f"Redshift : z = {z_snap:.3f}")

ds  = yt.load(snapfile)
obj = caesar.load(catfile)
ad  = ds.all_data()
a   = obj.simulation.scale_factor
print(f"scale_factor a = {a:.4f}  |  {len(obj.galaxies)} galaxies")

## 2. What black-hole / feedback fields are available?

SIMBA black holes are `PartType5`. We print the on-disk fields and the per-galaxy
BH attributes so the selection below can adapt to whatever your snapshot carries.

In [ ]:
# Particle fields that look BH / feedback related
bh_fields = [f for f in ds.field_list if f[0] == 'PartType5']
print("PartType5 (black hole) fields on disk:")
for f in bh_fields:
    print("   ", f)

# Per-galaxy BH info exposed by caesar
g0 = obj.galaxies[0]
print("\nExample galaxy mass keys:", list(getattr(g0, 'masses', {}).keys()))
for attr in ('bhlist', 'bhmdot', 'bh_fedd'):
    print(f"   gal.{attr:8s} present:", hasattr(g0, attr))

## 3. Find a galaxy that is (likely) launching a jet

We compute, per galaxy, the most-massive BH mass `M_BH`, its accretion rate
`Mdot`, and the Eddington ratio `f_Edd = Mdot / Mdot_Edd`. The jet-mode flag
follows SIMBA's criterion: `M_BH > 10^7.5 Msun` **and** `f_Edd < 0.2`.

Everything is wrapped in `try/except` — if a field is missing we fall back to
ranking by stellar mass (most massive galaxies host the biggest BHs). You can
always override with `TARGET_ID` at the bottom.

In [ ]:
from astropy import units as u, constants as const

def eddington_ratio(mbh_msun, mdot_msun_yr, eps=0.1):
    if not (mbh_msun > 0 and np.isfinite(mdot_msun_yr)):
        return np.nan
    L_edd = 1.26e38 * mbh_msun                      # erg/s
    mdot_edd = (L_edd * u.erg / u.s / (eps * const.c**2)).to(u.Msun / u.yr).value
    return mdot_msun_yr / mdot_edd

# Try to pull particle-level BH mass & accretion rate (best-effort units)
def bh_mass_mdot(gal):
    mbh = mdot = np.nan
    try:
        bhl = gal.bhlist
        if len(bhl) == 0:
            return mbh, mdot
        m = ds.arr(ad['PartType5', 'BH_Mass'][bhl], 'code_mass').in_units('Msun').value
        k = int(np.argmax(m))           # most massive BH in the galaxy
        mbh = float(m[k])
        try:
            md_ = ad['PartType5', 'BH_Mdot'][bhl].in_units('Msun/yr').value
        except Exception:
            # raw code units; SIMBA BH_Mdot code->Msun/yr factor is ~ unity-ish,
            # leave raw and let f_Edd be approximate
            md_ = np.asarray(ad['PartType5', 'BH_Mdot'][bhl])
        mdot = float(md_[k])
    except Exception as e:
        pass
    return mbh, mdot

rows = []
for i, g in enumerate(obj.galaxies):
    mstar = float(g.masses.get('stellar', np.nan))
    mbh, mdot = bh_mass_mdot(g)
    if not np.isfinite(mbh):
        mbh = float(g.masses.get('bh', np.nan))   # catalog fallback
    fedd = eddington_ratio(mbh, mdot)
    jet = (np.isfinite(mbh) and mbh > 10**7.5 and np.isfinite(fedd) and fedd < 0.2)
    rows.append((i, mstar, mbh, mdot, fedd, jet))

import numpy as np
arr = np.array([(r[0], r[1], r[2], r[3], r[4], r[5]) for r in rows], dtype=float)

jetmask = arr[:, 5] == 1
if jetmask.any():
    pool = arr[jetmask]
    pool = pool[np.argsort(-pool[:, 2])]      # by M_BH desc
    print("Jet-mode candidates (M_BH > 10^7.5, f_Edd < 0.2):")
    sel = pool
else:
    sel = arr[np.argsort(-np.nan_to_num(arr[:, 2]))]   # by M_BH desc
    print("No clean jet-mode galaxy flagged; ranking by M_BH instead:")

print(f"{'id':>5} {'logMstar':>9} {'logM_BH':>8} {'Mdot':>10} {'f_Edd':>9} {'jet?':>5}")
for r in sel[:10]:
    i = int(r[0])
    print(f"{i:5d} {np.log10(r[1]+1):9.2f} {np.log10(r[2]+1):8.2f} "
          f"{r[3]:10.3e} {r[4]:9.3e} {int(r[5]):5d}")

AUTO_TARGET = int(sel[0, 0])
print(f"\nAuto-selected TARGET_ID = {AUTO_TARGET}")

In [ ]:
# Override here if you want a specific galaxy:
TARGET_ID = AUTO_TARGET          # e.g. TARGET_ID = 6
gal = obj.galaxies[TARGET_ID]
print(f"Target galaxy {TARGET_ID}:  logMstar = {np.log10(gal.masses['stellar']):.2f}")

## 4. Helper: cache particle arrays + mass-weighted projection

These reuse the SPH-viewer mass-weighting trick already used in
`SingleRender.stream_plot`: to map a per-particle scalar `q`, we project `q*mass`
and `mass` and take the ratio (= mass-weighted line-of-sight mean of `q`).

- `region_ex=None`  -> galaxy members only (`gal.glist`)
- `region_ex=R`     -> **all gas within ~1.5R kpc** of the centre (CGM scale),
  which is where the jet deposits its energy.

In [ ]:
def get_gas_arrays(gal, region_ex=None, subtract_bulk=True):
    # Load gas pos/vel/mass/hsml/temp once and precompute radial velocity.
    center = gal.minpotpos.in_units('kpc').value * a

    if region_ex is None:
        idx = gal.glist
    else:
        allpos = ad['PartType0', 'Coordinates'].in_units('kpc').value * a
        d = np.linalg.norm(allpos - center, axis=1)
        idx = np.where(d < 1.5 * region_ex)[0]

    pos  = ad['PartType0', 'Coordinates'][idx].in_units('kpc').value * a
    vel  = ad['PartType0', 'Velocities'][idx].in_units('km/s').value * a
    mass = ad['PartType0', 'Masses'][idx].in_units('Msun').value
    hsml = ad['PartType0', 'SmoothingLength'][idx].in_units('kpc').value * a
    try:
        temp = ad['PartType0', 'Temperature'][idx].in_units('K').value
    except Exception:
        temp = np.full(len(mass), np.nan)

    if subtract_bulk:                      # remove the galaxy's bulk motion
        vel = vel - np.average(vel, axis=0, weights=mass)

    rel   = pos - center
    rnorm = np.linalg.norm(rel, axis=1)
    rhat  = rel / (rnorm[:, None] + 1e-30)
    vr    = np.sum(vel * rhat, axis=1)     # km/s, + = outflow, - = inflow

    return dict(pos=pos, vel=vel, mass=mass, hsml=hsml, temp=temp,
                vr=vr, center=center, L=gal.rotation['gas_L'])


def project_mass_weighted(g, quantity, ex, spos='edgeon', t=None, p=None):
    # Mass-weighted LOS projection of a per-particle scalar (or None for Sigma).
    t, p = find_rot_ax(g['L'], t, p, spos)
    R    = rotation_matrices_from_angles(t, p)
    cen  = g['center']
    pos_rot = (g['pos'] - cen) @ R.T + cen

    common = dict(hsml=g['hsml'], r='infinity', x=cen[0], y=cen[1], z=cen[2],
                  plot=False, extent=[-ex, ex, -ex, ex], logscale=False)
    qd = QuickView(pos_rot, g['mass'], **common)
    den = qd.get_image()
    extent = qd.get_extent()
    if quantity is None:                   # surface density map
        return den, den, extent
    num = QuickView(pos_rot, quantity * g['mass'], **common).get_image()
    out = np.divide(num, den, out=np.zeros_like(num), where=den > 0)
    return out, den, extent

## 5. The money plot: bipolar outflow triptych (edge-on)

Density (cavities) | Temperature (hot lobes) | **Radial velocity (bipolar outflow)**.

Look along the **minor axis** (vertical here): a jet shows as symmetric
red (outflow) lobes in the right panel, hot plumes in the middle, and evacuated
channels on the left.

In [ ]:
EXT_GALAXY = 30     # kpc half-width (resolved galaxy)
EXT_CGM    = 100    # kpc half-width (CGM environment)

EX   = EXT_CGM      # use CGM scale to catch the outflow; set EXT_GALAXY to zoom in
SPOS = 'edgeon'     # jets are launched along the minor axis -> view edge-on

g = get_gas_arrays(gal, region_ex=EX)

sigma, _, ext = project_mass_weighted(g, None,      EX, spos=SPOS)
temp,  den, _ = project_mass_weighted(g, g['temp'], EX, spos=SPOS)
vr,    _,   _ = project_mass_weighted(g, g['vr'],   EX, spos=SPOS)

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

axes[0].imshow(np.log10(sigma + sigma[sigma > 0].min()), origin='lower',
               extent=ext, cmap='bone')
axes[0].set_title(r'$\log\,\Sigma_{\rm gas}$')

with np.errstate(invalid='ignore'):
    logT = np.log10(np.where(temp > 0, temp, np.nan))
imT = axes[1].imshow(logT, origin='lower', extent=ext, cmap='inferno')
axes[1].set_title(r'$\log\,T$ [K]')
fig.colorbar(imT, ax=axes[1], fraction=0.046)

vmax = np.percentile(np.abs(vr[den > 0]), 99) if (den > 0).any() else 1.0
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
imV = axes[2].imshow(vr, origin='lower', extent=ext, cmap='RdBu_r', norm=norm)
axes[2].set_title(r'$v_r$ [km/s]   (red = outflow)')
fig.colorbar(imV, ax=axes[2], fraction=0.046)

for axx in axes:
    axx.set_xlabel('kpc'); axx.set_ylabel('kpc')
fig.suptitle(f'Galaxy {TARGET_ID}  |  z = {z_snap:.2f}  |  edge-on', y=1.02)
plt.tight_layout()
plt.show()

## 6. Velocity streamlines (your existing `SingleRender.stream_plot`)

A complementary view using the code already in the package. Streamlines diverging
out of the disk plane (edge-on) trace the outflow direction.

In [ ]:
sr = SingleRender(snapfile, catfile, TARGET_ID,
                  propr=('PartType0', 'Masses'), dim='Msun')
sr.stream_plot(ex=EXT_CGM, spos='edgeon', xl='kpc', yl='kpc')

## 7. "Real time" — interactive rotation

Arrays are cached in `g`, so the slider only re-projects (fast, no reload).
Spin the galaxy and watch the bipolar lobes sweep around. Requires `ipywidgets`.

In [ ]:
try:
    from ipywidgets import interact, FloatSlider, Dropdown

    t0, p0 = find_rot_ax(g['L'], None, None, 'edgeon')

    def show(azimuth=0.0, field='radial velocity', ex=EXT_CGM):
        if field == 'radial velocity':
            data, den, ext = project_mass_weighted(g, g['vr'], ex, t=t0, p=p0 + azimuth)
            vmx = np.percentile(np.abs(data[den > 0]), 99) if (den > 0).any() else 1
            kw = dict(cmap='RdBu_r',
                      norm=mcolors.TwoSlopeNorm(vmin=-vmx, vcenter=0, vmax=vmx))
        elif field == 'temperature':
            data, den, ext = project_mass_weighted(g, g['temp'], ex, t=t0, p=p0 + azimuth)
            with np.errstate(invalid='ignore'):
                data = np.log10(np.where(data > 0, data, np.nan))
            kw = dict(cmap='inferno')
        else:  # density
            data, den, ext = project_mass_weighted(g, None, ex, t=t0, p=p0 + azimuth)
            data = np.log10(data + data[data > 0].min())
            kw = dict(cmap='bone')
        fig, ax = plt.subplots(figsize=(7, 7))
        im = ax.imshow(data, origin='lower', extent=ext, **kw)
        fig.colorbar(im, ax=ax, fraction=0.046)
        ax.set_title(f'{field}  |  azimuth {azimuth:.0f} deg')
        ax.set_xlabel('kpc'); ax.set_ylabel('kpc')
        plt.show()

    interact(show,
             azimuth=FloatSlider(min=0, max=360, step=15, value=0),
             field=Dropdown(options=['radial velocity', 'temperature', 'density']),
             ex=FloatSlider(min=20, max=150, step=10, value=EXT_CGM))
except ImportError:
    print("ipywidgets not installed -> skipping interactive widget. "
          "Use the optional GIF cell below instead.")

## 8. (Optional) Rotation GIF of the outflow

Renders the radial-velocity map at a sequence of azimuths and stitches a GIF.
Heavier (one projection per frame) but needs no widgets.

In [ ]:
# Set to True to render the GIF
MAKE_GIF = False

if MAKE_GIF:
    import os
    from PIL import Image
    t0, p0 = find_rot_ax(g['L'], None, None, 'edgeon')
    frames = []
    outdir = os.path.join(os.getcwd(), 'output', sb.name, 'agn_feedback')
    os.makedirs(outdir, exist_ok=True)
    for k, az in enumerate(range(0, 360, 10)):
        vr, den, ext = project_mass_weighted(g, g['vr'], EXT_CGM, t=t0, p=p0 + az)
        vmx = np.percentile(np.abs(vr[den > 0]), 99)
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(vr, origin='lower', extent=ext, cmap='RdBu_r',
                  norm=mcolors.TwoSlopeNorm(vmin=-vmx, vcenter=0, vmax=vmx))
        ax.set_title(f'azimuth {az} deg'); ax.set_axis_off()
        fp = os.path.join(outdir, f'frame_{k:03d}.png')
        fig.savefig(fp, bbox_inches='tight', dpi=80); plt.close(fig)
        frames.append(Image.open(fp))
    gif = os.path.join(outdir, f'jet_vr_gal{TARGET_ID}_snap{SNAP}.gif')
    frames[0].save(gif, save_all=True, append_images=frames[1:], duration=120, loop=0)
    print("Saved", gif)

## 9. (Optional) Feedback *across snapshots* — the real time evolution

The most physical "feedback in real time": step through consecutive snapshots and
watch the outflow launch and propagate. Requires tracking the same galaxy across
snapshots (use your `merger_tracking` progenitor IDs). Sketch below — fill in the
per-snapshot galaxy IDs.

In [ ]:
# Sketch — adapt with progenitor IDs from your merger_tracking outputs.
EVOLVE = False
if EVOLVE:
    snap_to_id = {148: TARGET_ID, 149: TARGET_ID, 150: TARGET_ID}  # <- edit
    fig, axes = plt.subplots(1, len(snap_to_id), figsize=(7 * len(snap_to_id), 7))
    for ax, (snp, gid) in zip(np.atleast_1d(axes), sorted(snap_to_id.items())):
        ds_i  = yt.load(sb.get_sim_file(snp))
        obj_i = caesar.load(sb.get_caesar_file(snp))
        ad_i  = ds_i.all_data(); a_i = obj_i.simulation.scale_factor
        gi = obj_i.galaxies[gid]
        cen = gi.minpotpos.in_units('kpc').value * a_i
        allpos = ad_i['PartType0', 'Coordinates'].in_units('kpc').value * a_i
        idx = np.where(np.linalg.norm(allpos - cen, axis=1) < 1.5 * EXT_CGM)[0]
        pos  = allpos[idx]
        vel  = ad_i['PartType0', 'Velocities'][idx].in_units('km/s').value * a_i
        mass = ad_i['PartType0', 'Masses'][idx].in_units('Msun').value
        hsml = ad_i['PartType0', 'SmoothingLength'][idx].in_units('kpc').value * a_i
        vel -= np.average(vel, axis=0, weights=mass)
        rel = pos - cen; rhat = rel / (np.linalg.norm(rel, axis=1)[:, None] + 1e-30)
        vr = np.sum(vel * rhat, axis=1)
        gi_d = dict(pos=pos, mass=mass, hsml=hsml, center=cen, L=gi.rotation['gas_L'])
        vrmap, den, ext = project_mass_weighted(gi_d, vr, EXT_CGM, spos='edgeon')
        vmx = np.percentile(np.abs(vrmap[den > 0]), 99)
        ax.imshow(vrmap, origin='lower', extent=ext, cmap='RdBu_r',
                  norm=mcolors.TwoSlopeNorm(vmin=-vmx, vcenter=0, vmax=vmx))
        ax.set_title(f'snap {snp}'); ax.set_xlabel('kpc')
    plt.tight_layout(); plt.show()